[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aprendizaje-automatico-dc-uba-ar/material/blob/main/notebooks/lferrer/1_ec_vs_f1-published.ipynb)

# Comparison of EC with F-beta score

We compare both metrics on simulated data while varying the decision threshold. This is done for a binary classification task since F-beta is only defined for such case (generalizations to the multi-class case are just ugly hacks).

The simulated scores are obtained by sampling unidimensional features from Gaussian distributions for each class and then computing the posteriors for those samples (given the model).

In [ ]:
# Descomentar primera vez
# ! git clone https://github.com/luferrer/expected_cost

In [ ]:
import sys
sys.path.append(PATH_TO_EXPECTED_COST)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from expected_cost import ec, utils
from expected_cost.data import get_llks_for_multi_classif_task


In [ ]:
# Parameters for the simulated data. You can play with the first three parameters as much as you want.

N = 10000       # total number of samples
std = 0.20      # within-class standard deviation of the features, determines the difficulty of the problem
P0 = 0.5        # Prior for classes 0

num_targets = 2 # number of class (do not change, this nb deals with two-class problems only)
P1 = 1-P0
N0 = int(N*P0)
N1 = N - N0

# Generate the log-likelihoods and log posteriors

targets, _, llks = get_llks_for_multi_classif_task('gaussian_sim', priors=[P0,P1], sim_params={'feat_std': std}, N=N)
logpost = utils.llks_to_logpost(llks, [P0, P1])

# We take the logits as scores, since they look prettier
# than log posteriors when plotted.

scores = logpost[:,1]-logpost[:,0]


# Define various NEC with c12=1 and varying c21.

cbal = int(P0/P1) # c21 corresponding to the balanced error rate
cs = np.unique([1, cbal, 4, 16])
costs = dict([(c,[]) for c in cs])

# Define the range of thresholds to explore and determine the Bayes threshold for each cost function above.

thrs = np.arange(-11,8.01,0.1)

selthris = [] # Indices of the Bayes threshold for each cost
for c in cs:
    costs[c] = ec.CostMatrix([[0, 1], [c, 0]])
    bayes_thr = np.log(1/c)
    selthris.append(np.argmin(np.abs(thrs-bayes_thr)))



In [ ]:
fig1, axs1 = plt.subplots(1, 3, figsize=(14,3))

# Compute and save all metrics across a range of thresholds
# Rij is the fraction of samples of class i that was labelled as j. When i=s (star) it means
# "both classes". So, Rs1 is the fraction of samples labelled as class 1 by the system.

F1Ss, F2Ss, R10s, R01s, Rs1s = [], [], [], [], []
ECs = dict([(c,[]) for c in cs])

for thr in thrs:

    decisions = np.array(scores > thr, dtype=int)

    conf = ec.generalized_confusion_matrix(targets, decisions, num_targets=2, num_decisions=2)

    # Save the error rates for plotting
    K10 = conf[1,0]
    K01 = conf[0,1]
    R10s.append(K10/N1)
    R01s.append(K01/N0)
    Rs1s.append(np.sum(decisions==1)/len(decisions))

    for c in cs:
        ECs[c].append(ec.average_cost(targets, decisions, costs[c], adjusted=True))
    F1Ss.append(utils.Fscore(K10, K01, N0, N1))
    F2Ss.append(utils.Fscore(K10, K01, N0, N1, betasq=2))


# Plot distributions

ax = axs1[0]
c, h = utils.make_hist(targets, scores, nbins=50)
ax.plot(c,h[0]*P0,'k-', label='class 1')
ax.plot(c,h[1]*P1,'k--', label='class 2')

ax.set_xlabel('logit')
ylim = list(ax.get_ylim())
ylim[0] = 0
for selthri in selthris:
    thr = thrs[selthri]
    ax.plot([thr, thr],ylim, 'k:')
ax.set_ylim(ylim)
ax.legend()

# Plot comparison with Fbeta. Mark the threshold that optimizes F1 and also the threshold that gives
# the same EC as that other threshold, but a different F1.

maxy = 1.3
xlim = [-7, 7]

ax = axs1[1]
ax.plot(thrs, 1-np.array(F1Ss), 'g-', label=r'1-$\mathrm{F}_{1}$')
ax.plot(thrs, np.array(ECs[1]), 'r-', label=r'$\mathrm{NEC}_{%.0f}$'%1)
ax.plot(thrs, Rs1s, color='grey', linestyle='-', label=r'$R_{*2}$')

i1 = np.argmax(F1Ss)
ec_at_max = ECs[1][i1]
i2 = np.argsort(np.abs(ECs[1]-ec_at_max))[1]
ax.plot([thrs[i1], thrs[i1]],[0, maxy], 'k:')
ax.plot([thrs[i2], thrs[i2]],[0, maxy], 'k:')
ax.plot(xlim, [ec_at_max,ec_at_max],'k:')

# Plot various NEC, marking the Bayes threshold for each case.

ax = axs1[2]
styles = ['-','--',':','-.']
for i, c in enumerate(cs):
    ax.plot(thrs, ECs[c], 'r'+styles[i], label=r'$\mathrm{NEC}_{%.0f}$'%cs[i])
    thr = thrs[selthris[i]]
    ax.plot([thr, thr],[0, maxy], 'k:')

for i, ax in enumerate(axs1):
    ax.set_xlim(xlim)
    ax.legend(loc='upper right')
    if i>0:
        ax.set_ylim(0,maxy)
        ax.set_xlabel('thr')

